# Classify real Iris flowers

Install welt-client with its parquet extra, then connect your workspace in the first Python cell. Run the cells in order. These are real hosted requests: fitting retains a private training dataset and predictor. No extra helper file is needed. Outputs are deliberately not saved in this source.

In [ ]:
from welt import Client

with Client() as client:
    client.connect()


We follow six steps: get data → inspect → split → fit → predict → evaluate. Iris150 rows/four measurements/three encoded classes; CC BY4.0, R. A. Fisher (1936), UCI Iris DOI10.24432/C56C76. Seed9 stratified120/30 split.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from welt import Client, Classifier

# 1. Get a real table from Welt's research inventory.
dataset_id = "asset-d839644b6de36fc5cafdda18c5daeca3"
version = "d839644b6de36fc5cafdda18c5daeca3284a5ce8bb6169aff2fcb3fbb4a12021"
path = Path(mkdtemp(prefix="welt-iris-")) / "iris.parquet"
with Client() as client:
    info = client.research_dataset(dataset_id)
    print("Dataset:", info["name"], "— License:", info["license"])
    for notice in info["license_notices"]:
        print(notice)
    client.download_research_dataset(dataset_id, path, version=version)
data = pd.read_parquet(path)
target = "__target__#0"  # The target recorded in this research snapshot.



In [ ]:
# 2. Look at the rows before modelling.
print(data.head())
print("Rows and columns:", data.shape)
data.info()
print(data[target].value_counts())



In [ ]:
# 3. Set aside test rows before fitting; never train on their answers.
train, test = train_test_split(
    data, test_size=0.2, random_state=9, stratify=data[target])
query = test.drop(columns=target)
y_test = test[target]
print("Training rows:", len(train), "— Test rows:", len(test))



In [ ]:
# 4. Create a model and train it on the training table.
model = Classifier(model="tabicl-v2", random_state=9)
model.fit(train, target=target)



In [ ]:
# 5. Predict the held-out rows, without passing their target column.
predictions = model.predict(query)
print(pd.DataFrame({"actual": y_test.to_numpy(), "predicted": predictions}).head())



In [ ]:
# 6. Compare predictions with the answers we kept aside.
print("Test accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions, zero_division=0))
print("Keep this predictor ID:", model.predictor_id_)


Read the actual versus predicted rows and the metrics above. Scores depend on your held-out split and model version; no minimum score is promised. Keep the test set for assessment rather than repeated tuning.

In [ ]:
print("Pinned model version:", model.model_version_)
print("Reusable predictor:", model.predictor_id_)